# Ejercicio AR — Esfera virtual sobre marcador planar

**Objetivo:** crear un efecto de realidad aumentada en el que el usuario desplace una esfera virtual hacia posiciones marcadas con el ratón. La esfera rueda físicamente sobre el plano definido por el marcador.

El ejercicio aplica los conceptos vistos en teoría:
1. Calibración intrínseca con `Kfov`
2. Estimación de pose con `solvePnP`
3. Homografía del plano z=0
4. Proyección de puntos 3D con la matriz de cámara
5. Rodamiento físicamente correcto con la fórmula de Rodrigues

## 1. Marco teórico

### Calibración intrínseca

La matriz de calibración $K$ relaciona las coordenadas 3D de cámara con los píxeles de la imagen. Se aproxima a partir del campo de visión horizontal (FOV):

$$K = \begin{pmatrix} f \cdot w/2 & 0 & w/2 \\ 0 & f \cdot w/2 & h/2 \\ 0 & 0 & 1 \end{pmatrix}, \quad f = \frac{1}{\tan(\text{FOV}/2)}$$

### Estimación de pose

Dado el marcador en L con 6 vértices conocidos en coordenadas del mundo $(X_i, Y_i, 0)$ y sus correspondencias en la imagen $(u_i, v_i)$, `solvePnP` (internamente `Pose` de umucv) estima la rotación $R$ y traslación $\mathbf{t}$ que minimizan el error de reproyección:

$$M = K\,[R\,|\,\mathbf{t}]$$

### Homografía del plano z=0

Para los puntos del plano $z=0$, la tercera columna de $M$ no contribuye. Se extrae la homografía:

$$H = M[:, [0, 1, 3]]$$

Un click en $(u, v)$ se mapea al mundo con:

$$\tilde{\mathbf{X}} = H^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}, \quad (X, Y) = \tilde{\mathbf{X}}_{1:2} / \tilde{\mathbf{X}}_3$$

### Rodamiento

Al moverse un vector $\Delta\mathbf{p}$ sobre el plano, la esfera gira alrededor del eje perpendicular al movimiento:

$$\text{eje} = \hat{z} \times \Delta\mathbf{p}, \quad \theta = \frac{|\Delta\mathbf{p}|}{r}, \quad dR = \text{Rodrigues}(\text{eje}\,\cdot\,\theta)$$

La orientación acumulada se actualiza: $R \leftarrow dR \cdot R$

## 2. Marcador planar

El marcador es una figura en L con 6 vértices en el plano $z=0$, con dimensiones $1 \times 1$ en unidades del mundo.

In [ ]:
import cv2 as cv
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
from umucv.htrans import htrans, Pose, Kfov, sepcam
from umucv.util import lineType
from umucv.contours import extractContours, redu

marker = np.array([
    [0.0, 0.0, 0],
    [0.0, 1.0, 0],
    [0.5, 1.0, 0],
    [0.5, 0.5, 0],
    [1.0, 0.5, 0],
    [1.0, 0.0, 0],
])

fig, ax = plt.subplots(figsize=(4, 4))
loop = np.vstack([marker[:, :2], marker[0, :2]])
ax.fill(loop[:, 0], loop[:, 1], color='#333333', alpha=0.8)
ax.plot(loop[:, 0], loop[:, 1], 'o-', color='orange')
for i, (x, y, _) in enumerate(marker):
    ax.annotate(f'$P_{i}$', (x, y), textcoords='offset points', xytext=(6, 4))
ax.set_aspect('equal'); ax.set_title('Marcador en el plano z=0'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Detección del marcador y estimación de pose

`extractContours` aplica umbralización de Otsu y `approxPolyDP` para obtener polígonos. Se busca el contorno con exactamente 6 vértices. Para cada rotación cíclica del contorno se llama a `solvePnP` y se queda la pose con menor error RMS de reproyección.

In [ ]:
def polygons(cs, n, prec=2):
    for eps in [prec, prec*2, prec*4, prec*8, prec*16]:
        rs = [redu(c, eps) for c in cs]
        found = [r for r in rs if len(r) == n]
        if found:
            return found
    return []

def rots(c):
    return [np.roll(c, k, 0) for k in range(len(c))]

def bestPose(K, view, model):
    poses = [Pose(K, v.astype(float), model) for v in rots(view)]
    return sorted(poses, key=lambda p: p.rms)[0]

frame = cv.imread('figura3.jpeg')
H_fr, W_fr = frame.shape[:2]
K = Kfov((W_fr, H_fr), 60)

g    = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
cs   = extractContours(g, minarea=2, reduprec=3)
good = polygons(cs, 6, 3)

pose = None
for c in good:
    p = bestPose(K, c, marker)
    if p.rms < 8:
        pose = p
        break

print(f'Marcador detectado: {pose is not None}')
if pose:
    print(f'RMS reproyección: {pose.rms:.2f} px')
    print(f'Traslación t = {pose.t.round(3)}')

## 4. Malla triangulada de la esfera

La esfera se discretiza en `n_lat × n_lon` bandas de latitud/longitud. Cada celda se divide en 2 triángulos. El patrón de damero negro/blanco permite ver la rotación visualmente.

In [ ]:
def sphere_mesh(r=0.12, n_lat=12, n_lon=18):
    verts = []
    for i in range(n_lat + 1):
        th = np.pi * i / n_lat
        for j in range(n_lon):
            ph = 2 * np.pi * j / n_lon
            verts.append([r * np.sin(th) * np.cos(ph),
                          r * np.sin(th) * np.sin(ph),
                          r * np.cos(th)])
    verts = np.array(verts)

    tris, colors = [], []
    for i in range(n_lat):
        for j in range(n_lon):
            j2 = (j + 1) % n_lon
            v0, v1 = i * n_lon + j,  i * n_lon + j2
            v2, v3 = (i+1)*n_lon + j, (i+1)*n_lon + j2
            c = np.array([30, 30, 30], np.float32) if (i + j) % 2 == 0 \
                else np.array([245, 245, 245], np.float32)
            tris.append([v0, v2, v1]); colors.append(c)
            tris.append([v1, v2, v3]); colors.append(c)
    return verts, np.array(tris, int), np.array(colors, np.float32)

SPHERE_R = 0.12
sphere_verts, sphere_tris, sphere_tri_colors = sphere_mesh(r=SPHERE_R)

print(f'Vértices: {len(sphere_verts)}')
print(f'Triángulos: {len(sphere_tris)}')

fig = plt.figure(figsize=(5, 5))
ax  = fig.add_subplot(111, projection='3d')
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
polys      = [[sphere_verts[i] for i in tri] for tri in sphere_tris[::4]]
colors_mpl = [(c[0]/255, c[1]/255, c[2]/255, 0.8) for c in sphere_tri_colors[::4]]
col = Poly3DCollection(polys, facecolors=colors_mpl, edgecolors='none')
ax.add_collection3d(col)
ax.set_xlim(-SPHERE_R, SPHERE_R); ax.set_ylim(-SPHERE_R, SPHERE_R); ax.set_zlim(-SPHERE_R, SPHERE_R)
ax.set_title('Malla triangulada de la esfera'); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 5. Render de la esfera

Para proyectar la esfera sobre la imagen se usa la matriz de cámara $M$ estimada con la pose del marcador.
Los vértices 3D de la esfera se proyectan con `htrans(M, world_v)` y cada triángulo se rellena con `fillPoly`.
Se utiliza sombreado y ordenación por profundidad (técnicas estándar de renderizado 3D por software) para dar sensación de volumen.

In [ ]:
LIGHT_DIR = np.array([1.0, -1.0, 2.0])
LIGHT_DIR /= la.norm(LIGHT_DIR)
AMBIENT = 0.35

def draw_sphere(frame, M, sphere_pos, sphere_R):
    _, _, C = sepcam(M)
    C = C.flatten()

    rot_v   = sphere_verts @ sphere_R.T
    world_v = rot_v + sphere_pos + np.array([0, 0, SPHERE_R])
    n_world = rot_v / (la.norm(rot_v, axis=1, keepdims=True) + 1e-9)
    img_v   = htrans(M, world_v)   # proyección 3D → imagen

    visible = []
    for k, tri in enumerate(sphere_tris):
        v0, v1, v2 = tri
        n = n_world[v0] + n_world[v1] + n_world[v2]
        n /= la.norm(n) + 1e-9
        centroid = (world_v[v0] + world_v[v1] + world_v[v2]) / 3
        view = C - centroid
        view /= la.norm(view) + 1e-9
        if n @ view <= 0:
            continue
        shade = AMBIENT + (1 - AMBIENT) * max(0.0, n @ LIGHT_DIR)
        color = (sphere_tri_colors[k] * shade).clip(0, 255).astype(int)
        depth = la.norm(centroid - C)
        visible.append((depth, tri, color))

    visible.sort(key=lambda t: -t[0])

    for _, tri, color in visible:
        pts = img_v[tri].astype(int)
        cv.fillPoly(frame, [pts], tuple(int(c) for c in color), cv.LINE_AA)

print('Función draw_sphere definida.')

## 6. Homografía del plano z=0 y rodamiento

Un click en $(u,v)$ se proyecta al plano del mundo usando $H = M[:,\,[0,1,3]]$.
La esfera se mueve suavemente hacia el destino y rueda con la fórmula de Rodrigues.

In [ ]:
M = pose.M
H = M[:, [0, 1, 3]]   # homografía del plano z=0

# Ejemplo: click en el centro de la imagen
u_click, v_click = W_fr // 2, H_fr // 2
uv  = np.array([u_click, v_click, 1.0])
xyw = la.inv(H) @ uv
xyw /= xyw[2]
target = np.array([xyw[0], xyw[1], 0.0])
print(f'Click ({u_click},{v_click}) → mundo ({target[0]:.3f}, {target[1]:.3f})')

# Rodamiento: eje perpendicular al movimiento, ángulo = distancia / radio
SPEED = 0.8
dt    = 0.04
pos   = np.array([0.5, 0.5, 0.])
sph_R = np.eye(3)
delta = target - pos
dist  = la.norm(delta)
step  = min(SPEED * dt, dist)
move  = delta / dist * step
axis  = np.cross([0., 0., 1.], move)
axis /= la.norm(axis)
angle = step / SPHERE_R
dR    = cv.Rodrigues(axis * angle)[0]
sph_R = dR @ sph_R
print(f'Paso de movimiento: {step:.4f} unidades de mundo')
print(f'Ángulo de giro:     {np.degrees(angle):.2f}°')

## 7. Resultado

Esfera renderizada sobre `figura3.jpeg` con la pose estimada del marcador.

In [ ]:
result = cv.imread('resultado_ar.png')

plt.figure(figsize=(7, 9))
plt.imshow(cv.cvtColor(result, cv.COLOR_BGR2RGB))
plt.title('Esfera virtual sobre el marcador planar')
plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Programa interactivo

El script completo [`ar_mouse.py`](ar_mouse.py) integra todo lo anterior en un bucle de captura en tiempo real:

```bash
python ar_mouse.py
# o con imagen estática para pruebas:
python ar_mouse.py --dev=figura3.jpeg --loop
```

**Controles:**
- **Click izquierdo** sobre el plano del marcador → la esfera rueda suavemente hasta ese punto
- **q / ESC** → salir

La esfera se mantiene sobre el plano inclinado del marcador gracias a que su posición 3D $(X, Y, r)$ se proyecta con la matriz $M = K[R|\mathbf{t}]$ estimada en cada frame.